# Gaussian check: fBm H = 0.5 (white), Scat2 + Scat4: $D_k = D_{KL}(p\|q_k)$ per group of potentials

$p \propto \exp(-\sum_j \theta_j^T\phi_j)$, $A_k = \theta_k^T\phi_k$, $q_k \propto p\,e^{A_k}$;
$D_k = \log E_p[e^{A_k - E_p A_k}] \ge 0$ (nats). Code sign: $\log p = +\Theta\cdot\phi$, so $A_k = -C_k$.
Groups: summed energy of the group (never a sum of single $D_k$). $L_6$ is never removed.

**Assumptions** (flagged): $E_p$ is estimated on the GENERATED signals (law $\rho_1$ of the SDE endpoint) while
$\theta = \Theta_{reg}(t=1)$; $\hat D_k$ is exact only if $\rho_1 = p_{\Theta_{reg}}$. Error bar = max(std over seeds, bootstrap std).

**What the numbers mean:** $\hat D_k \le D_k$ (Jensen: $E\log\bar W \le \log E\bar W$, on average) and $I_k \le D_k$, with
$I_k = D_{KL}(p\|q_k^*)$ the information of term $k$ (other $\theta$ refitted). $\hat D_k$ and $I_k$ are not ordered.
Groups with degenerate weights (fewer than 100 signals effectively share the weight, ESS < 100, or $\hat D_k$ still rising with N, in any seed) are shown as
LOWER BOUNDS $D_k \ge \hat D_k$ (open circles / "$\ge$"), no error bar: "removing this term alone, without
refit, changes p strongly", NOT "this term carries much information". Gaussian rule: $N \gg e^{\mathrm{Var}(A)/2} = e^{D_{KL}(q_k\|p)}$
samples needed (column log10_N_needed). $\frac12\mathrm{Var}(A_k)$ is the 2nd-cumulant approximation only.
Heavy computation: `kl_groups/launch_kl_groups.sh` (cache_energies.py + compute_H.py); this notebook only reads
`results/<TAG>/H_fbm.csv`.

In [1]:
import os, sys
from pathlib import Path
TAG = 'run1'          # results/<TAG>
os.environ['KL_RESULTS'] = str(Path('results') / TAG)
sys.path.insert(0, '.')
import plots as P
import matplotlib.pyplot as plt
%matplotlib inline
DS = 'fbm'
df, agg = P.load(DS, Path(os.environ['KL_RESULTS']))
print(df.seed.unique(), 'seeds |', df.N.iloc[0], 'samples per seed')

[300 301 302 303 304 305 306 307 308 309] seeds | 15000 samples per seed


## $D_k$ per group, one marker per seed, with $\frac12$Var alongside

In [ ]:
P.bars(agg, title=DS); plt.show()

## $D_k$ per single potential, sorted

In [ ]:
for fam in ('Fourth_Order_Mod2_Real', 'Fourth_Order_Mod2_Imag'):
    if agg.family.str.contains(fam).any():
        P.singles(agg, fam, title=fam); plt.show()
P.singles(agg, '.', title='all families'); plt.show()

## $D_{j,j'}$ (all $l$) vs $|j-j'|$ and $(j,j')$ heatmap
Hypothesis to TEST: decay with $|j-j'|$, possibly dominated by $|j-j'|=1$.

In [ ]:
for fam in ('Fourth_Order_Mod2_Real', 'Fourth_Order_Mod2_Imag'):
    for ang in ('all',):
        if ((agg.kind == 'pair') & agg.family.str.contains(fam) & (agg.label == ang)).any():
            d = P.pairs(agg, fam, title=f'{fam} [{ang}]', angle=ang); plt.show()
            print(d.groupby('dj').H.agg(['mean', 'max', 'count']).to_string())

## Diagnostics: ESS of the weights $e^{A_k}$, $\hat D_k$ vs N for the top contributors

In [ ]:
P.diagnostics(agg, DS, Path(os.environ['KL_RESULTS'])); plt.show()

## Summary table (ratio = $D_k / \frac12$Var)

In [ ]:
P.summary(agg, kinds=('category', 'family')).round(4)

In [ ]:
P.summary(agg, kinds=('pair',)).head(30).round(4)